In [6]:
import xarray as xr
import numpy as np
import xesmf as xe
from dask.diagnostics import ProgressBar

In [2]:
cmems_phy = xr.open_zarr("./resources/copernicus_marine_service/cmems_obs-mob_glo_phy_my_0.125deg_P1M-m.zarr", consolidated=True)

In [3]:
cmems_phy

<xarray.Dataset> Size: 10GB
Dimensions:    (depth: 50, latitude: 220, longitude: 157, time: 145)
Coordinates:
  * depth      (depth) int16 100B 0 5 10 15 20 25 ... 3500 4000 4500 5000 5500
  * latitude   (latitude) float32 880B -59.94 -59.81 -59.69 ... -32.69 -32.56
  * longitude  (longitude) float32 628B -69.56 -69.44 -69.31 ... -50.19 -50.06
  * time       (time) datetime64[ns] 1kB 2012-12-01 2013-01-01 ... 2024-12-01
Data variables:
    mlotst     (time, latitude, longitude) float64 40MB dask.array<chunksize=(145, 220, 157), meta=np.ndarray>
    so         (time, depth, latitude, longitude) float64 2GB dask.array<chunksize=(145, 50, 220, 157), meta=np.ndarray>
    to         (time, depth, latitude, longitude) float64 2GB dask.array<chunksize=(145, 50, 220, 157), meta=np.ndarray>
    ugo        (time, depth, latitude, longitude) float64 2GB dask.array<chunksize=(145, 50, 220, 157), meta=np.ndarray>
    vgo        (time, depth, latitude, longitude) float64 2GB dask.array<chunksize=(145, 50, 220, 157), meta=np.ndarray>
    zo         (time, depth, latitude, longitude) float64 2GB dask.array<chunksize=(145, 50, 220, 157), meta=np.ndarray>
Attributes:
    history:                   2025-10-13 20:10:31 ARMOR3D REP - TSHUV Global...
    institution:               CLS
    title:                     ARMOR3D REP - TSHUVMld Global Ocean Observatio...
    Conventions:               CF-1.0
    copernicusmarine_version:  2.2.5

In [ ]:
#whe need compute() to get the values from dask arrays (lazy evaluation), due to being backed by zarr file
lat_diff = cmems_phy['latitude'].diff(dim='latitude')
lat_diff_vals = lat_diff.compute()
lon_diff = cmems_phy['longitude'].diff(dim='longitude')
lon_diff_vals = lon_diff.compute()
print("Latitude spacing unique values:", np.unique(lat_diff_vals))
print("Longitude spacing unique values:", np.unique(lon_diff_vals))
print("Bounds")
print(cmems_phy['latitude'].min().compute().item(), cmems_phy['latitude'].max().compute().item())
print(cmems_phy['longitude'].min().compute().item(), cmems_phy['longitude'].max().compute().item())

Latitude spacing unique values: [0.125]
Longitude spacing unique values: [0.125]
Regrided bounds:
-59.9375 -32.5625
-69.5625 -50.0625


In [7]:
#now regrid to regular grid for the model
min_lon, min_lat, max_lon, max_lat = [-69.61, -60., -50., -32.45] 


res = 0.125
new_lons = np.arange(min_lon, max_lon + res, res)
new_lats = np.arange(min_lat, max_lat + res, res)

# Create a target grid as xarray Dataset
ds_tgt = xr.Dataset({
    'lat': (['lat'], new_lats),
    'lon': (['lon'], new_lons)})

regridder = xe.Regridder(cmems_phy, ds_tgt, 'conservative')
with ProgressBar():
    cmems_phy_regridded = regridder(cmems_phy).compute()


[########################################] | 100% Completed | 67.27 ss


In [9]:
lat_diff = cmems_phy_regridded['lat'].diff(dim='lat')
lat_diff_vals = lat_diff.compute()
lon_diff = cmems_phy_regridded['lon'].diff(dim='lon')
lon_diff_vals = lon_diff.compute()
print("Latitude spacing unique values:", np.unique(lat_diff_vals))
print("Longitude spacing unique values:", np.unique(lon_diff_vals))
print("Regrided bounds:")
print(cmems_phy_regridded['lat'].min().compute().item(), cmems_phy_regridded['lat'].max().compute().item())
print(cmems_phy_regridded['lon'].min().compute().item(), cmems_phy_regridded['lon'].max().compute().item())

Latitude spacing unique values: [0.125]
Longitude spacing unique values: [0.125]
Regrided bounds:
-60.0 -32.375
-69.61 -49.985
